In [1]:
pip install scikit-learn pandas joblib nltk

  Using cached nltk-3.10.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
Using cached nltk-3.10.0-py3-none-any.whl (1.7 MB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
# Download stopwords list and WordNet for lemmatization
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mohit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mohit\AppData\Roaming\nltk_data...


True

In [3]:
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Initialize lemmatizer and load English stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase and remove HTML break tags
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    
    # 2. Remove punctuation, numbers, and non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 3. Tokenize, remove stop words, and lemmatize
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(cleaned_words)

print("✅ Preprocessing pipeline defined successfully.")

✅ Preprocessing pipeline defined successfully.


In [4]:
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import joblib

# Set paths based on your workspace layout
DATASET_DIR = os.path.join("..", "dataset") # Adjusted assuming you run inside the 'notebooks' folder
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# 1. Load IMDb dataset
print("Loading IMDb Sentiment Dataset...")
imdb_df = pd.read_csv(os.path.join(DATASET_DIR, "IMDB Dataset.csv"))

# Assume columns are named 'review' and 'sentiment'
# Clean the text (This might take a minute depending on CPU compute)
print("Cleaning text data...")
imdb_df['cleaned_text'] = imdb_df['review'].apply(clean_text)

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    imdb_df['cleaned_text'], imdb_df['sentiment'], test_size=0.2, random_state=42
)

# 3. Vectorization (Convert text to sparse matrix)
print("Vectorizing text using TF-IDF...")
sentiment_vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 2))
X_train_vec = sentiment_vectorizer.fit_transform(X_train)
X_test_vec = sentiment_vectorizer.transform(X_test)

# 4. Train LinearSVC
print("Training LinearSVC model...")
sentiment_model = LinearSVC(C=1.0, random_state=42)
sentiment_model.fit(X_train_vec, y_train)

# 5. Evaluate
y_pred = sentiment_model.predict(X_test_vec)
print("\n--- Sentiment Model Performance ---")
print(classification_report(y_test, y_pred))

# 6. Save Model & Vectorizer
joblib.dump(sentiment_model, os.path.join(MODELS_DIR, "sentiment_model.pkl"))
joblib.dump(sentiment_vectorizer, os.path.join(MODELS_DIR, "sentiment_vectorizer.pkl"))
print("💾 Saved sentiment model and vectorizer to models/ directory.")

Loading IMDb Sentiment Dataset...
Cleaning text data...
Vectorizing text using TF-IDF...
Training LinearSVC model...

--- Sentiment Model Performance ---
              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      4961
    positive       0.90      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000

💾 Saved sentiment model and vectorizer to models/ directory.


**Vibe triainig

In [ ]:
from sklearn.naive_bayes import ComplementNB

# 1. Load Vibe dataset
print("Loading Vibe Dataset...")
vibe_df = pd.read_csv(os.path.join(DATASET_DIR, "vibe_dataset.csv"))

print("Cleaning vibe text data...")
vibe_df['cleaned_text'] = vibe_df['text'].apply(clean_text)

# 2. Split Data
X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    vibe_df['cleaned_text'], vibe_df['vibe'], test_size=0.15, random_state=42
)

# 3. Vectorization (Separate vectorizer optimized for emotional vocabulary)
print("Vectorizing vibe text...")
vibe_vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_v_vec = vibe_vectorizer.fit_transform(X_train_v)
X_test_v_vec = vibe_vectorizer.transform(X_test_v)

# 4. Train ComplementNB
print("Training ComplementNB model...")
vibe_model = ComplementNB(alpha=1.0)
vibe_model.fit(X_train_v_vec, y_train_v)

# 5. Evaluate
y_pred_v = vibe_model.predict(X_test_v_vec)
print("\n--- Vibe Model Performance ---")
print(classification_report(y_test_v, y_pred_v))

# 6. Save Model & Vectorizer
joblib.dump(vibe_model, os.path.join(MODELS_DIR, "vibe_model.pkl"))
joblib.dump(vibe_vectorizer, os.path.join(MODELS_DIR, "vibe_vectorizer.pkl"))
print("💾 Saved vibe model and vectorizer to models/ directory.")

Loading Vibe Dataset...
Cleaning vibe text data...
Vectorizing vibe text...
Training ComplementNB model...

--- Vibe Model Performance ---
              precision    recall  f1-score   support

   Emotional       0.48      0.55      0.51       343
    Exciting       0.47      0.42      0.44       683
 Frustrating       0.53      0.60      0.57      1387
       Funny       0.58      0.57      0.58       465
Heartwarming       0.67      0.67      0.67      2406
Mind-blowing       0.44      0.37      0.41       984
       Scary       0.48      0.47      0.48       142

    accuracy                           0.56      6410
   macro avg       0.52      0.52      0.52      6410
weighted avg       0.56      0.56      0.56      6410

💾 Saved vibe model and vectorizer to models/ directory.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report
from scipy.sparse import hstack
import joblib
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

DATASET_DIR = os.path.join("..", "dataset") 
MODELS_DIR = os.path.join("..", "models")

# Load existing sentiment assets to generate our metadata engine
sentiment_model = joblib.load(os.path.join(MODELS_DIR, "sentiment_model.pkl"))
sentiment_vectorizer = joblib.load(os.path.join(MODELS_DIR, "sentiment_vectorizer.pkl"))

# 1. ENHANCED CLEANING: Retain critical emotional indicators
lemmatizer = WordNetLemmatizer()
base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never', 'but', 'very', 'too', 'only', 'what', 'why'}
custom_stopwords = base_stopwords - negation_words

def advanced_clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    cleaned = [lemmatizer.lemmatize(w) for w in words if w not in custom_stopwords]
    return " ".join(cleaned)

# 2. Load and clean Vibe dataset
print("Loading and preparing datasets...")
vibe_df = pd.read_csv(os.path.join(DATASET_DIR, "vibe_dataset.csv"))
vibe_df['cleaned_text'] = vibe_df['text'].apply(advanced_clean_text)

# 3. Generate Sentiment Meta-Features 
print("Generating Meta-Features via pre-trained Sentiment Model...")
# Feed the text through the sentiment vectorizer + model
sent_vecs = sentiment_vectorizer.transform(vibe_df['cleaned_text'])
# LinearSVC decision function acts as a structural continuous sentiment metric
sentiment_scores = sentiment_model.decision_function(sent_vecs).reshape(-1, 1)

# 4. Train-Test Split (Including the scores)
indices = np.arange(len(vibe_df))
X_train_text, X_test_text, y_train, y_test, idx_train, idx_test = train_test_split(
    vibe_df['cleaned_text'], vibe_df['vibe'], indices, test_size=0.15, random_state=42
)

# 5. Text Vectorization
print("Vectorizing text...")
vibe_vectorizer = TfidfVectorizer(max_features=35000, ngram_range=(1, 3), min_df=3)
X_train_tfidf = vibe_vectorizer.fit_transform(X_train_text)
X_test_tfidf = vibe_vectorizer.transform(X_test_text)

# 6. Feature Stacking (Horizontally merge text patterns with sentiment metrics)
X_train_combined = hstack([X_train_tfidf, sentiment_scores[idx_train]])
X_test_combined = hstack([X_test_tfidf, sentiment_scores[idx_test]])

# 7. Model Training with Regularization Tuning
print("Training Meta-Feature Stacked LinearSVC...")
base_svc = LinearSVC(C=0.3, class_weight='balanced', random_state=42, max_iter=3000)
vibe_model = CalibratedClassifierCV(base_svc, cv=3)
vibe_model.fit(X_train_combined, y_train)

# 8. Evaluation
y_pred = vibe_model.predict(X_test_combined)
print("\n--- Meta-Feature Stacked Vibe Model Performance ---")
print(classification_report(y_test, y_pred))

# 9. Save all structures
joblib.dump(vibe_model, os.path.join(MODELS_DIR, "vibe_modelv2.pkl"))
joblib.dump(vibe_vectorizer, os.path.join(MODELS_DIR, "vibe_vectorizerv2.pkl"))
print("💾 Meta-stacked models exported successfully.")

Loading and preparing datasets...
Generating Meta-Features via pre-trained Sentiment Model...
Vectorizing text...
Training Meta-Feature Stacked LinearSVC...

--- Meta-Feature Stacked Vibe Model Performance ---
              precision    recall  f1-score   support

   Emotional       0.60      0.46      0.52       343
    Exciting       0.57      0.35      0.44       683
 Frustrating       0.57      0.64      0.60      1387
       Funny       0.68      0.63      0.65       465
Heartwarming       0.64      0.77      0.70      2406
Mind-blowing       0.51      0.41      0.45       984
       Scary       0.55      0.32      0.41       142

    accuracy                           0.61      6410
   macro avg       0.59      0.51      0.54      6410
weighted avg       0.60      0.61      0.60      6410

💾 Meta-stacked models exported successfully.


Loading and preparing datasets...
Building Dual-Engine Feature Union (Word + Char)...
Vectorizing text matrix...
Training SGDClassifier (Modified Huber)...

--- Ultimate Dual-Engine Vibe Model Performance ---
              precision    recall  f1-score   support

   Emotional       0.41      0.63      0.50       343
    Exciting       0.45      0.45      0.45       683
 Frustrating       0.59      0.59      0.59      1387
       Funny       0.61      0.75      0.67       465
Heartwarming       0.73      0.62      0.67      2406
Mind-blowing       0.46      0.47      0.47       984
       Scary       0.42      0.57      0.48       142

    accuracy                           0.58      6410
   macro avg       0.53      0.58      0.55      6410
weighted avg       0.60      0.58      0.59      6410

💾 Maximum-yield Scikit-Learn models exported successfully.


In [13]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import classification_report
import joblib
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

DATASET_DIR = os.path.join("..", "dataset") 
MODELS_DIR = os.path.join("..", "models")

# 1. Cleaning Pipeline
lemmatizer = WordNetLemmatizer()
base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never', 'but', 'very', 'too', 'only', 'what', 'why'}
custom_stopwords = base_stopwords - negation_words

def advanced_clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    cleaned = [lemmatizer.lemmatize(w) for w in words if w not in custom_stopwords]
    return " ".join(cleaned)

print("Loading and preparing datasets...")
vibe_df = pd.read_csv(os.path.join(DATASET_DIR, "vibe_dataset.csv"))
vibe_df['cleaned_text'] = vibe_df['text'].apply(advanced_clean_text)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    vibe_df['cleaned_text'], vibe_df['vibe'], test_size=0.15, random_state=42
)

# 2. Build the Dual-Engine Feature Union
print("Building Dual-Engine Feature Union (Word + Char)...")

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3), 
    min_df=3,
    max_features=25000,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', # Character n-grams inside word boundaries
    ngram_range=(2, 4),
    min_df=5,
    max_features=25000,
    sublinear_tf=True
)

# Combine both engines mathematically
vibe_vectorizer = FeatureUnion([
    ('word', word_vectorizer),
    ('char', char_vectorizer)
])

# 3. Vectorize Text
print("Vectorizing text matrix...")
X_train_vec = vibe_vectorizer.fit_transform(X_train_text)
X_test_vec = vibe_vectorizer.transform(X_test_text)

# 4. Train SGDClassifier 
print("Training SGDClassifier (Modified Huber)...")
vibe_model = SGDClassifier(
    loss='modified_huber',  # Natively supports predict_proba()
    penalty='l2',
    alpha=1e-4,
    class_weight='balanced',
    max_iter=3000,
    random_state=42,
    early_stopping=True
)

vibe_model.fit(X_train_vec, y_train)

# 5. Evaluate
y_pred = vibe_model.predict(X_test_vec)
print("\n--- Ultimate Dual-Engine Vibe Model Performance ---")
print(classification_report(y_test, y_pred))

# 6. Save Model
joblib.dump(vibe_model, os.path.join(MODELS_DIR, "vibe_model.pkl"))
joblib.dump(vibe_vectorizer, os.path.join(MODELS_DIR, "vibe_vectorizer.pkl"))
print("💾 Maximum-yield Scikit-Learn models exported successfully.")

Loading and preparing datasets...
Building Dual-Engine Feature Union (Word + Char)...
Vectorizing text matrix...
Training SGDClassifier (Modified Huber)...

--- Ultimate Dual-Engine Vibe Model Performance ---
              precision    recall  f1-score   support

   Emotional       0.47      0.65      0.55       268
    Exciting       0.48      0.56      0.52       455
 Frustrating       0.66      0.67      0.67      1371
       Funny       0.76      0.84      0.80       448
Heartwarming       0.79      0.69      0.74      2111
Mind-blowing       0.52      0.54      0.53       899
       Scary       0.56      0.62      0.59       154

    accuracy                           0.66      5706
   macro avg       0.61      0.65      0.63      5706
weighted avg       0.67      0.66      0.66      5706

💾 Maximum-yield Scikit-Learn models exported successfully.


**SANDBOX

In [14]:
import joblib
import os

MODELS_DIR = os.path.join("..", "models")

# Load compiled assets
s_model = joblib.load(os.path.join(MODELS_DIR, "sentiment_model.pkl"))
s_vec = joblib.load(os.path.join(MODELS_DIR, "sentiment_vectorizer.pkl"))
v_model = joblib.load(os.path.join(MODELS_DIR, "vibe_model.pkl"))
v_vec = joblib.load(os.path.join(MODELS_DIR, "vibe_vectorizer.pkl"))

def test_review(review_text):
    cleaned = clean_text(review_text)
    
    # Predict Sentiment
    s_vec_data = s_vec.transform([cleaned])
    sentiment_prediction = s_model.predict(s_vec_data)[0]
    
    # Predict Vibe
    v_vec_data = v_vec.transform([cleaned])
    vibe_prediction = v_model.predict(v_vec_data)[0]
    
    print(f"\nReview: \"{review_text}\"")
    print(f"🔮 Sentiment -> {sentiment_prediction}")
    print(f"🎭 Vibe      -> {vibe_prediction}")

# Try it out!
test_review("Absolutely mind-bending masterpiece! The plot twists left me completely speechless and confused in the best way possible.")
test_review("Honestly, this was an incredibly boring and frustrating waste of time. The pacing dragged forever.")


Review: "Absolutely mind-bending masterpiece! The plot twists left me completely speechless and confused in the best way possible."
🔮 Sentiment -> positive
🎭 Vibe      -> Heartwarming

Review: "Honestly, this was an incredibly boring and frustrating waste of time. The pacing dragged forever."
🔮 Sentiment -> negative
🎭 Vibe      -> Frustrating
